# Bluesky Local Text Preparation

Local-only preparation of Bluesky post text using phase6 diagnostic raw/hydrated files.


## 1. Identify and Load Local Source Files
- No Snowflake
- No firehose/API reruns
- Canonical sources fixed to phase6 run root


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'nlp' / 'post_normalization.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/nlp/post_normalization.py')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.post_normalization import load_jsonl_gz, prepare_bluesky_posts_from_files

RAW_PATH = ROOT / 'data/phase6_diagnostic_20260401/raw_posts/cap_20260401T184939Z_85c99020/raw_posts_000001.jsonl.gz'
HYDRATED_PATH = ROOT / 'data/phase6_diagnostic_20260401/hydrated_posts/hyd_20260401T185109Z_97b40e6f/hydrated_posts_000001.jsonl.gz'

for p in (RAW_PATH, HYDRATED_PATH):
    if not p.exists():
        raise FileNotFoundError(f'Missing required local source file: {p}')

print('repo_root:', ROOT)
print('raw_path:', RAW_PATH)
print('hydrated_path:', HYDRATED_PATH)


In [ ]:
raw_rows = load_jsonl_gz(RAW_PATH)
hydrated_rows = load_jsonl_gz(HYDRATED_PATH)
print('raw_rows:', len(raw_rows))
print('hydrated_rows:', len(hydrated_rows))
print('raw_top_keys:', sorted(raw_rows[0].keys()))
print('hydrated_top_keys:', sorted(hydrated_rows[0].keys()))


## 2. Apply Deterministic Text Preparation
Extraction policy: `record.text` primary, empty-string fallback; one row per `uri`, hydrated preferred.


In [ ]:
prepared_df = prepare_bluesky_posts_from_files(
    raw_paths=[RAW_PATH],
    hydrated_paths=[HYDRATED_PATH],
    source_run_tag='phase6_diagnostic_20260401',
)
print('prepared_rows:', len(prepared_df))
prepared_df.head(5)


## 3. Before/After Examples


In [ ]:
prepared_df[['uri', 'post_text_raw', 'post_text_clean', 'post_text_alnum', 'text_source']].head(12)


## 4. Pattern-Focused Inspection


In [ ]:
def show_subset(title: str, mask):
    subset = prepared_df.loc[mask, [
        'uri', 'post_text_raw', 'post_text_clean', 'has_hashtag', 'has_url', 'has_mention',
        'has_special_chars', 'has_non_ascii'
    ]].head(10)
    print(f'\n## {title} (rows={int(mask.sum())})')
    if subset.empty:
        print('No examples in this dataset slice.')
    else:
        display(subset)

show_subset('Hashtag-heavy', prepared_df['has_hashtag'])
show_subset('URL-heavy', prepared_df['has_url'])
show_subset('Mention-heavy', prepared_df['has_mention'])
show_subset('Punctuation-heavy', prepared_df['has_special_chars'])
show_subset('Non-ASCII / noisy', prepared_df['has_non_ascii'])


## 5. Helper Field Validation


In [ ]:
summary = {
    'rows': int(len(prepared_df)),
    'text_source_counts': prepared_df['text_source'].value_counts().to_dict(),
    'has_hashtag': int(prepared_df['has_hashtag'].sum()),
    'has_url': int(prepared_df['has_url'].sum()),
    'has_mention': int(prepared_df['has_mention'].sum()),
    'has_special_chars': int(prepared_df['has_special_chars'].sum()),
    'has_non_ascii': int(prepared_df['has_non_ascii'].sum()),
    'token_count_describe': prepared_df['post_token_count'].describe().to_dict(),
    'char_count_describe': prepared_df['post_char_count'].describe().to_dict(),
}
summary


## 6. Save Prepared Outputs


In [ ]:
OUT_FULL = ROOT / 'local/reference_snapshots/bluesky/bluesky_posts_text_prepared.parquet'
OUT_SAMPLE_PARQUET = ROOT / 'data/samples/bluesky_posts_text_prepared_sample.parquet'
OUT_SAMPLE_CSV = ROOT / 'data/samples/bluesky_posts_text_prepared_sample.csv'

OUT_FULL.parent.mkdir(parents=True, exist_ok=True)
OUT_SAMPLE_PARQUET.parent.mkdir(parents=True, exist_ok=True)

prepared_df.to_parquet(OUT_FULL, index=False)
sample_df = prepared_df.sort_values('uri').head(min(1000, len(prepared_df))).copy()
sample_df.to_parquet(OUT_SAMPLE_PARQUET, index=False)
sample_df.to_csv(OUT_SAMPLE_CSV, index=False)

print('wrote:', OUT_FULL)
print('wrote:', OUT_SAMPLE_PARQUET)
print('wrote:', OUT_SAMPLE_CSV)
print('sample_rows:', len(sample_df))


## 7. Readiness Note
- Text is now in stable local prepared form with deterministic helper fields.
- Next phase can begin local topic extraction (no matching logic yet).
